In [2]:
import pandas as pd

# File paths for each dataset
file_paths = {
    "EUR/USD": "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/prepared_eur_usd_data.csv",
    "GBP/USD": "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/prepared_gbp_usd_data.csv",
    "USD/CAD": "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/prepared_usd_cad_data.csv",
    "USD/CNY": "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/prepared_usd_cny_data.csv",
    "USD/JPY": "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/prepared_usd_jpy_data.csv",
}

# Load datasets
forex_data = {pair: pd.read_csv(path) for pair, path in file_paths.items()}

# Ensure the 'Date' column is in datetime format for merging
for pair, df in forex_data.items():
    df['Date'] = pd.to_datetime(df['Date'])
    forex_data[pair] = df

# Display the first few rows of one dataset
print(forex_data["EUR/USD"].head())


        Date     Close      High       Low      Open  Price Change     SMA_3  \
0 2014-01-28  1.367465  1.368940  1.363180  1.367596     -0.000971  1.368389   
1 2014-01-29  1.365505  1.368599  1.360450  1.365430     -0.001434  1.367255   
2 2014-01-30  1.365859  1.366214  1.356061  1.365915      0.000260  1.366276   
3 2014-01-31  1.355877  1.356239  1.348109  1.355859     -0.007308  1.362414   
4 2014-02-03  1.348818  1.351960  1.347931  1.348818     -0.005206  1.356851   

      SMA_5    SMA_10  Price Difference  ...      MACD  MACD Signal  \
0  1.363061  1.361155          0.005761  ... -0.002035    -0.003133   
1  1.364979  1.361021          0.008149  ... -0.001699    -0.002846   
2  1.367306  1.361613          0.010153  ... -0.001388    -0.002555   
3  1.364700  1.361021          0.008131  ... -0.001925    -0.002429   
4  1.360705  1.360627          0.004029  ... -0.002886    -0.002520   

       SlowK      SlowD  BB_upper  BB_middle  BB_lower       ATR  \
0  77.371776  59.081904 

In [4]:
# Start with one dataset
merged_data = forex_data["EUR/USD"]

# Merge with the other datasets
for pair, df in forex_data.items():
    if pair != "EUR/USD":
        merged_data = pd.merge(merged_data, df, on="Date", suffixes=("", f"_{pair.replace('/', '_')}"))

print(merged_data.head())


        Date     Close      High       Low      Open  Price Change     SMA_3  \
0 2014-01-28  1.367465  1.368940  1.363180  1.367596     -0.000971  1.368389   
1 2014-01-29  1.365505  1.368599  1.360450  1.365430     -0.001434  1.367255   
2 2014-01-30  1.365859  1.366214  1.356061  1.365915      0.000260  1.366276   
3 2014-01-31  1.355877  1.356239  1.348109  1.355859     -0.007308  1.362414   
4 2014-02-03  1.348818  1.351960  1.347931  1.348818     -0.005206  1.356851   

      SMA_5    SMA_10  Price Difference  ...  MACD_USD_JPY  \
0  1.363061  1.361155          0.005761  ...     -0.522420   
1  1.364979  1.361021          0.008149  ...     -0.522674   
2  1.367306  1.361613          0.010153  ...     -0.591184   
3  1.364700  1.361021          0.008131  ...     -0.601587   
4  1.360705  1.360627          0.004029  ...     -0.648033   

   MACD Signal_USD_JPY  SlowK_USD_JPY  SlowD_USD_JPY  BB_upper_USD_JPY  \
0            -0.346810      22.203315      40.873515        105.922172  

In [6]:
# Load sentiment analysis data
sentiment_data = pd.read_csv("C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/sentiment_analysis/data/forex_news_articles.csv") 
sentiment_data.rename(columns={"date": "Date"}, inplace=True) 
sentiment_data['Date'] = pd.to_datetime(sentiment_data['Date'])  # Ensure datetime format

# Merge with the combined forex dataset
final_data = pd.merge(merged_data, sentiment_data, on="Date", how="left")

# Inspect the final dataset
print(final_data.head())


        Date     Close      High       Low      Open  Price Change     SMA_3  \
0 2014-01-28  1.367465  1.368940  1.363180  1.367596     -0.000971  1.368389   
1 2014-01-29  1.365505  1.368599  1.360450  1.365430     -0.001434  1.367255   
2 2014-01-30  1.365859  1.366214  1.356061  1.365915      0.000260  1.366276   
3 2014-01-30  1.365859  1.366214  1.356061  1.365915      0.000260  1.366276   
4 2014-01-31  1.355877  1.356239  1.348109  1.355859     -0.007308  1.362414   

      SMA_5    SMA_10  Price Difference  ...  SlowK_USD_JPY  SlowD_USD_JPY  \
0  1.363061  1.361155          0.005761  ...      22.203315      40.873515   
1  1.364979  1.361021          0.008149  ...      20.928489      27.989917   
2  1.367306  1.361613          0.010153  ...      23.822112      22.317972   
3  1.367306  1.361613          0.010153  ...      23.822112      22.317972   
4  1.364700  1.361021          0.008131  ...      25.276654      23.342418   

   BB_upper_USD_JPY  BB_middle_USD_JPY  BB_lower_U

In [7]:
# Fill missing values (example: forward fill)
final_data.fillna(method="ffill", inplace=True)

# Alternatively, drop rows with missing values
# final_data.dropna(inplace=True)


C:\Users\vamsh\AppData\Local\Temp\ipykernel_7444\456665589.py:2: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  final_data.fillna(method="ffill", inplace=True)


In [8]:
final_data.to_csv("C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/final_forex_data.csv", index=False)
print("Final dataset saved!")


Final dataset saved!


Merging currency indexes

In [1]:
import pandas as pd

currency_files = {
    "USD" : "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/usd.csv",
    "GBP" : "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/gbp.csv",
    "CAD" : "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/cad.csv",
    "CNY" : "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/cny.csv",
    "JPY" : "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/JPY.csv",
    "EUR" : "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/euro.csv"
}


currency_data = {}

for currency, file_path in currency_files.items():
    df = pd.read_csv(file_path, parse_dates=["Date"])
    df = df[['Date', 'Close', 'High', 'Low', 'Open', 'Volume']]
    df = df.rename(columns={
        "Close": f"{currency}_Close",
        "High": f"{currency}_High",
        "Low": f"{currency}_Low",
        "Open": f"{currency}_Open",
        "Volume": f"{currency}_Volume"
    })
    df.set_index("Date", inplace=True)
    currency_data[currency] = df

merged_data = pd.concat(currency_data.values(), axis=1, join="inner")
merged_data.reset_index(inplace=True)
merged_data.index = merged_data.index + 1
print(merged_data.head())
merged_data.to_csv("C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/data/currency index/merged_currency_indexes.csv", index=True)



        Date  USD_Close   USD_High    USD_Low   USD_Open  USD_Volume  \
1 2014-01-06  80.650002  80.910004  80.540001  80.870003         0.0   
2 2014-01-07  80.830002  80.949997  80.599998  80.680000         0.0   
3 2014-01-08  81.040001  81.169998  80.830002  80.910004         0.0   
4 2014-01-09  81.010002  81.190002  80.849998  81.070000         0.0   
5 2014-01-10  80.660004  81.139999  80.529999  80.949997         0.0   

     GBP_Close     GBP_High      GBP_Low     GBP_Open  ...     JPY_Close  \
1  6730.700195  6752.000000  6714.600098  6730.700195  ...  15908.879883   
2  6755.500000  6768.899902  6718.100098  6730.700195  ...  15814.370117   
3  6721.799805  6755.500000  6713.399902  6755.500000  ...  16121.450195   
4  6691.299805  6746.399902  6679.299805  6721.799805  ...  15880.330078   
5  6739.899902  6769.899902  6691.299805  6691.299805  ...  15912.059570   

       JPY_High       JPY_Low      JPY_Open   JPY_Volume   EUR_Close  \
1  16164.009766  15864.440430  16147.5